In [1]:
import os, gc
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

AUTOTUNE = tf.data.AUTOTUNE

In [2]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print("✅ GPU Available")
    for gpu in gpus:
        print("🔥", gpu)
else:
    print("❌ No GPU found")

✅ GPU Available
🔥 PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
🔥 PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')


In [10]:
class CFG:
    DATA_PATH = "/kaggle/input/datasets/obulisainaren/multi-cancer/Multi Cancer/Multi Cancer"
    IMG_SIZE = (224,224)
    BATCH_SIZE = 128
    CHUNK_SIZE = 20000   # 🔥 key parameter
    EPOCHS_PER_CHUNK = 1
    SEED = 42

In [13]:
def load_data(path):
    fp, ct, sc = [], [], []
    
    ct_map = {c:i for i,c in enumerate(sorted(os.listdir(path)))}
    sc_map = {}
    sc_counter = 0

    for c in ct_map:
        for s in os.listdir(os.path.join(path,c)):
            if s not in sc_map:
                sc_map[s] = sc_counter
                sc_counter += 1
            
            for img in os.listdir(os.path.join(path,c,s)):
                fp.append(os.path.join(path,c,s,img))
                ct.append(ct_map[c])
                sc.append(sc_map[s])

    return fp, ct, sc, ct_map, sc_map

file_paths, ct_labels, sc_labels, ct_to_idx, sc_to_idx = load_data(CFG.DATA_PATH)

In [14]:
train_fp, val_fp, train_ct, val_ct, train_sc, val_sc = train_test_split(
    file_paths, ct_labels, sc_labels,
    test_size=0.2, stratify=ct_labels, random_state=CFG.SEED
)

In [15]:
def build_dataset(paths, ct, sc):
    ds = tf.data.Dataset.from_tensor_slices((paths, ct, sc))

    def load(p, ct, sc):
        img = tf.io.read_file(p)
        img = tf.io.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, CFG.IMG_SIZE)
        img = tf.cast(img, tf.float32)/255.0

        return img, {
            "cancer_type": ct,
            "subclass": sc
        }

    ds = ds.map(load, num_parallel_calls=4)
    ds = ds.shuffle(5000)
    ds = ds.batch(CFG.BATCH_SIZE)
    return ds.prefetch(AUTOTUNE)

In [6]:
def build_model():
    base = tf.keras.applications.MobileNetV3Small(
        input_shape=(224,224,3),
        include_top=False,
        weights="imagenet"
    )
    base.trainable = False

    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.Dense(128, activation="relu")(x)

    ct_out = tf.keras.layers.Dense(len(ct_to_idx), activation="softmax", name="cancer_type")(x)
    sc_out = tf.keras.layers.Dense(len(sc_to_idx), activation="softmax", name="subclass")(x)

    model = tf.keras.Model(
        inputs=base.input,
        outputs={"cancer_type": ct_out, "subclass": sc_out}
    )

    model.compile(
        optimizer="adam",
        loss={
            "cancer_type": "sparse_categorical_crossentropy",
            "subclass": "sparse_categorical_crossentropy"
        },
        metrics={
    "cancer_type": ["accuracy"],
    "subclass": ["accuracy"]
}
    )

    return model

model = build_model()

4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [7]:
# =========================================
# IMPROVED CHUNK TRAINING LOOP
# =========================================

import gc

num_samples = len(train_fp)
indices = np.arange(num_samples)

# 🔥 shuffle ONCE before training
np.random.shuffle(indices)

# ✅ build validation dataset ONCE
val_ds = build_dataset(val_fp, val_ct, val_sc)

for chunk_id, start in enumerate(range(0, num_samples, CFG.CHUNK_SIZE)):
    end = min(start + CFG.CHUNK_SIZE, num_samples)

    print(f"\n🔥 Chunk {chunk_id+1} | Samples: {start} → {end}")

    chunk_idx = indices[start:end]

    chunk_fp = [train_fp[i] for i in chunk_idx]
    chunk_ct = [train_ct[i] for i in chunk_idx]
    chunk_sc = [train_sc[i] for i in chunk_idx]

    # ✅ build dataset
    train_ds = build_dataset(chunk_fp, chunk_ct, chunk_sc)

    # ✅ train
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=5,
        verbose=1
    )

    # 🧹 CLEAN MEMORY (VERY IMPORTANT)
    del train_ds, end, chunk_idx, chunk_fp, chunk_ct, chunk_sc
    gc.collect()

    # 🔥 optional: clear TF graph every few chunks
    if chunk_id % 3 == 0:
        tf.keras.backend.clear_session()


🔥 Chunk 1 | Samples: 0 → 20000
Epoch 1/5


I0000 00:00:1774189280.865947      98 service.cc:152] XLA service 0x7d2ff8003f10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774189280.865991      98 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1774189280.865997      98 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1774189282.120150      98 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1774189290.378338      98 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


157/157 ━━━━━━━━━━━━━━━━━━━━ 196s 1s/step - cancer_type_accuracy: 0.8099 - cancer_type_loss: 0.4993 - loss: 1.8604 - subclass_accuracy: 0.5123 - subclass_loss: 1.3610 - val_cancer_type_accuracy: 0.8193 - val_cancer_type_loss: 0.4929 - val_loss: 1.8609 - val_subclass_accuracy: 0.5036 - val_subclass_loss: 1.3676
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 78s 454ms/step - cancer_type_accuracy: 0.8059 - cancer_type_loss: 0.5009 - loss: 1.8628 - subclass_accuracy: 0.5209 - subclass_loss: 1.3619 - val_cancer_type_accuracy: 0.7220 - val_cancer_type_loss: 0.7323 - val_loss: 2.3626 - val_subclass_accuracy: 0.4112 - val_subclass_loss: 1.6313
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 70s 401ms/step - cancer_type_accuracy: 0.8147 - cancer_type_loss: 0.4999 - loss: 1.8682 - subclass_accuracy: 0.5108 - subclass_loss: 1.3682 - val_cancer_type_accuracy: 0.8184 - val_cancer_type_loss: 0.4948 - val_loss: 1.8953 - val_subclass_accuracy: 0.4868 - val_subclass_loss: 1.4016
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 6

KeyboardInterrupt: 

In [ ]:
model.save("/kaggle/working/chunk_trained_model.h5")
model.save("/kaggle/working/f_trained_model.keras")

In [3]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print("🔥 GPUs:", gpus)

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

strategy = tf.distribute.MirroredStrategy()
print("🔥 Devices:", strategy.num_replicas_in_sync)

🔥 GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
🔥 Devices: 2


I0000 00:00:1774191555.999411      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774191556.005504      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [4]:
MODEL_PATH = "/kaggle/working/f_trained_model.keras"

with strategy.scope():
    model = tf.keras.models.load_model(MODEL_PATH)

print("✅ Model loaded for fine-tuning")
model.summary()

✅ Model loaded for fine-tuning


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv (Conv2D)       │ (None, 112, 112,  │        432 │ rescaling[0][0]   │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_bn             │ (None, 112, 112,  │         64 │ conv[0][0]        │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 112, 112,  │          0 │ conv_bn[0][0]     │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 113, 113,  │          0 │ activation[0][0]  │
│ (ZeroPadding2D)     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 56, 56,    │        144 │ expanded_conv_de… │
│ (DepthwiseConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 56, 56,    │         64 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 56, 56,    │          0 │ expanded_conv_de… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 16)  │          0 │ re_lu[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 8)   │        136 │ expanded_conv_sq… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 8)   │          0 │ expanded_conv_sq… │
│ (ReLU)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 16)  │        144 │ expanded_conv_sq… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 1, 1, 16)  │          0 │ expanded_conv_sq… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 1, 1, 16)  │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 1, 1, 16)  │          0 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 56, 56,    │          0 │ re_lu[0][0],      │
│ (Multiply)          │ 16)               │            │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 56, 56,    │        256 │ expanded_conv_sq

 Total params: 1,095,606 (4.18 MB)

 Trainable params: 78,242 (305.63 KB)

 Non-trainable params: 939,120 (3.58 MB)

 Optimizer params: 78,244 (305.64 KB)

In [ ]:
import tensorflow as tf

MODEL_PATH = "/kaggle/working/f_trained_model.keras"  # change if needed

model = tf.keras.models.load_model(MODEL_PATH)

print("✅ Model loaded successfully")
model.summary()



In [5]:
# Unfreeze last few layers for fine-tuning
for layer in model.layers[-20:]:
    layer.trainable = True

print("🔥 Last 20 layers unfrozen")

🔥 Last 20 layers unfrozen


In [6]:
with strategy.scope():
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-5),  # 👈 LOWER LR
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

print("✅ Model recompiled for fine-tuning")

✅ Model recompiled for fine-tuning


In [7]:
BATCH_SIZE = 32 * strategy.num_replicas_in_sync
print("🔥 New batch size:", BATCH_SIZE)

🔥 New batch size: 64


In [16]:
EPOCHS = 5

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

print("🔥 Fine-tuning complete")

NameError: name 'train_ds' is not defined

In [ ]:
loss, acc = model.evaluate(test_ds)

print("\n📊 AFTER FINE-TUNING")
print("🔥 Loss:", loss)
print("🔥 Accuracy:", acc)